# Reading H1 Receiver Observation Data

This notebook demonstrates how to read HDF5 files produced by the H1 Receiver scheduler,
inspect the observation metadata, and plot a waterfall diagram and mean spectrum.

Data is stored as **linear power** (not dB), which preserves radiometric accuracy for
calibration workflows (Y-factor, system temperature, antenna temperature).
Conversion to dB is done only for display.

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timezone
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Select and open an HDF5 file

List available `.h5` files in the data folder and pick one to examine.

In [ ]:
# List available data files
data_dir = Path('data')
h5_files = sorted(data_dir.glob('*.h5')) if data_dir.exists() else sorted(Path('.').glob('*.h5'))

print(f'Found {len(h5_files)} HDF5 file(s):')
for i, f in enumerate(h5_files):
    print(f'  [{i}] {f.name}')

# Select which file to open (change this index as needed)
FILE_INDEX = -1  # -1 = most recent
h5_path = h5_files[FILE_INDEX]
print(f'\nSelected: {h5_path}')

## 2. Read metadata and datasets

In [ ]:
hf = h5py.File(h5_path, 'r')

# Read datasets
freq_hz = hf['frequency_hz'][:]
freq_mhz = freq_hz / 1e6
spectra_linear = hf['spectra_linear'][:]
timestamps = hf['timestamps'][:]
integration_times = hf['integration_times'][:]

n_spectra, n_channels = spectra_linear.shape

# Read all metadata attributes
attrs = dict(hf.attrs)

print(f'Datasets loaded: {n_spectra} spectra x {n_channels} channels')
print(f'Frequency range: {freq_mhz[0]:.3f} - {freq_mhz[-1]:.3f} MHz')
print(f'Time span: {timestamps[-1] - timestamps[0]:.1f} seconds')
print(f'Total integration: {np.sum(integration_times):.1f} seconds')

## 3. Display observation metadata

Show all attributes stored in the HDF5 file, including receiver settings and
observation context from the scheduler.

In [ ]:
# Receiver metadata (always present)
print('=== Receiver Settings ===')
print(f"  SDR type:          {attrs.get('sdr_type', 'N/A')}")
print(f"  Center frequency:  {attrs.get('center_freq_hz', 0) / 1e6:.3f} MHz")
print(f"  Sample rate:       {attrs.get('sample_rate_hz', 0) / 1e6:.3f} MHz")
print(f"  FFT size:          {attrs.get('fft_size', 'N/A')}")
print(f"  Gain:              {attrs.get('gain_db', 'N/A')} dB")
print(f"  Integration time:  {attrs.get('nominal_integration_time', 'N/A')} s")
print(f"  Created:           {attrs.get('created', 'N/A')}")

# Observation metadata (present when launched from scheduler)
if 'obs_name' in attrs:
    print()
    print('=== Observation Context ===')
    print(f"  Name:              {attrs.get('obs_name')}")
    print(f"  Scheduled:         {attrs.get('start_date', '')} {attrs.get('start_time', '')}")
    print(f"  Duration:          {attrs.get('duration_minutes', 'N/A')} minutes")
    print(f"  Calibrator:        {'ON' if attrs.get('calibrator', 0) else 'OFF'}")

    coord_sys = attrs.get('coord_system', '')
    if coord_sys == 'object':
        print(f"  Target:            {attrs.get('object_name', 'N/A')} (solar system object)")
    elif coord_sys == 'radec':
        ra = attrs.get('coord1_deg', 0) + attrs.get('coord1_min', 0)/60 + attrs.get('coord1_sec', 0)/3600
        dec = attrs.get('coord2_deg', 0) + attrs.get('coord2_min', 0)/60 + attrs.get('coord2_sec', 0)/3600
        print(f"  Target:            RA {ra:.4f}h, Dec {dec:.4f}\u00b0")
    elif coord_sys == 'galactic':
        l = attrs.get('coord1_deg', 0) + attrs.get('coord1_min', 0)/60 + attrs.get('coord1_sec', 0)/3600
        b = attrs.get('coord2_deg', 0) + attrs.get('coord2_min', 0)/60 + attrs.get('coord2_sec', 0)/3600
        print(f"  Target:            Gal l={l:.2f}\u00b0, b={b:.2f}\u00b0")
    elif coord_sys == 'altaz':
        alt = attrs.get('coord1_deg', 0) + attrs.get('coord1_min', 0)/60 + attrs.get('coord1_sec', 0)/3600
        az = attrs.get('coord2_deg', 0) + attrs.get('coord2_min', 0)/60 + attrs.get('coord2_sec', 0)/3600
        print(f"  Target:            Alt {alt:.2f}\u00b0, Az {az:.2f}\u00b0")
else:
    print('\n  (No scheduler metadata - receiver was run standalone)')

## 4. Build a descriptive title from metadata

Used by the plots below.

In [ ]:
def make_title():
    """Build a plot title from observation metadata."""
    parts = []
    if 'obs_name' in attrs:
        parts.append(str(attrs['obs_name']))
    if attrs.get('coord_system') == 'object':
        parts.append(str(attrs.get('object_name', '')).capitalize())
    if attrs.get('calibrator', 0):
        parts.append('(CAL ON)')
    parts.append(str(attrs.get('created', '')[:19]))
    return ' \u2014 '.join(parts) if parts else h5_path.name

def to_db(linear):
    """Convert linear power to dB, handling zeros."""
    return 10 * np.log10(np.maximum(linear, 1e-30))

title = make_title()
print(f'Plot title: {title}')

## 5. Waterfall plot

Shows how the spectrum evolves over time. Each row is one integration period.
Linear power is converted to dB for display.

In [ ]:
# Convert to dB for display
spectra_db = to_db(spectra_linear)

fig, ax = plt.subplots(figsize=(12, 6))

# Convert timestamps to minutes from start
t_minutes = (timestamps - timestamps[0]) / 60.0

# Use robust colour limits from the data
vmin = np.percentile(spectra_db, 5)
vmax = np.percentile(spectra_db, 95)

im = ax.imshow(
    spectra_db,
    aspect='auto',
    origin='lower',
    extent=[freq_mhz[0], freq_mhz[-1], t_minutes[0], t_minutes[-1]],
    cmap='viridis',
    vmin=vmin,
    vmax=vmax,
)

cb = fig.colorbar(im, ax=ax, label='Power (dB)')
ax.set_xlabel('Frequency (MHz)')
ax.set_ylabel('Time (minutes from start)')
ax.set_title(f'Waterfall \u2014 {title}')

# Mark the H1 rest frequency
ax.axvline(x=1420.405, color='r', linestyle='--', linewidth=0.8, alpha=0.7, label='H I 1420.405 MHz')
ax.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

## 6. Mean spectrum

Average all integrations **in linear power** to produce a single high-SNR spectrum,
displayed in linear power units. This is the correct representation for radiometric
work (Y-factor, system temperature, antenna temperature calibration).

In [ ]:
# Average in linear power domain
mean_linear = np.mean(spectra_linear, axis=0)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(freq_mhz, mean_linear, linewidth=0.5, color='steelblue')
ax.axvline(x=1420.405, color='r', linestyle='--', linewidth=0.8, alpha=0.7, label='H I 1420.405 MHz')

ax.set_xlabel('Frequency (MHz)')
ax.set_ylabel('Power (linear)')
ax.set_title(f'Mean Spectrum ({n_spectra} integrations) \u2014 {title}')
ax.legend(loc='upper right', fontsize=8)
ax.grid(True, alpha=0.3)

# Annotate with key metadata
info_text = f"SDR: {attrs.get('sdr_type', '?')}  |  Gain: {attrs.get('gain_db', '?')} dB"
info_text += f"  |  BW: {attrs.get('sample_rate_hz', 0)/1e6:.1f} MHz  |  {n_channels} ch"
total_integration = np.sum(integration_times)
info_text += f"  |  Total integration: {total_integration:.1f}s"
ax.text(0.01, 0.02, info_text, transform=ax.transAxes, fontsize=7,
        color='gray', verticalalignment='bottom')

plt.tight_layout()
plt.show()

## 7. Zoomed view around the hydrogen line

Zoom into a narrow window around 1420.405 MHz to see the H I emission detail.

In [ ]:
# Zoom to +/- 0.5 MHz around H1
h1_freq = 1420.405
zoom_width = 0.5  # MHz either side
mask = (freq_mhz >= h1_freq - zoom_width) & (freq_mhz <= h1_freq + zoom_width)

if mask.sum() > 0:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True,
                                    gridspec_kw={'height_ratios': [1, 2]})

    # Zoomed mean spectrum (linear power)
    ax1.plot(freq_mhz[mask], mean_linear[mask], linewidth=0.8, color='steelblue')
    ax1.axvline(x=h1_freq, color='r', linestyle='--', linewidth=0.8, alpha=0.7)
    ax1.set_ylabel('Power (linear)')
    ax1.set_title(f'H I Region Detail \u2014 {title}')
    ax1.grid(True, alpha=0.3)

    # Zoomed waterfall (dB for colour mapping)
    zoomed_spectra = spectra_db[:, mask]
    zoomed_freq = freq_mhz[mask]
    vmin_z = np.percentile(zoomed_spectra, 5)
    vmax_z = np.percentile(zoomed_spectra, 95)

    im = ax2.imshow(
        zoomed_spectra,
        aspect='auto',
        origin='lower',
        extent=[zoomed_freq[0], zoomed_freq[-1], t_minutes[0], t_minutes[-1]],
        cmap='viridis',
        vmin=vmin_z,
        vmax=vmax_z,
    )
    fig.colorbar(im, ax=ax2, label='Power (dB)')
    ax2.axvline(x=h1_freq, color='r', linestyle='--', linewidth=0.8, alpha=0.7)
    ax2.set_xlabel('Frequency (MHz)')
    ax2.set_ylabel('Time (minutes from start)')

    plt.tight_layout()
    plt.show()
else:
    print(f'H I line at {h1_freq} MHz is outside the observed band ({freq_mhz[0]:.3f} - {freq_mhz[-1]:.3f} MHz)')

In [ ]:
hf.close()